In [134]:
from datasets import load_dataset, load_from_disk
from sklearn.model_selection import train_test_split
from nltk.tokenize import word_tokenize
from nltk.stem.wordnet import WordNetLemmatizer
from nltk.corpus import wordnet
import numpy as np
from scipy.sparse import csr_matrix
from nltk.util import bigrams
import nltk
nltk.download('punkt_tab')
nltk.download('wordnet')
lemmatizer = WordNetLemmatizer()

/home/muhannd/venv/lib/python3.12/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/muhannd/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/muhannd/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /home/muhannd/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [105]:
try:
    ds = load_from_disk("./datasets/emotion_dataset")
except FileNotFoundError:
    ds = load_dataset("avsolatorio/mteb-emotion-avs_triplets")
    ds.save_to_disk("./datasets/emotion_dataset")

In [106]:
ds['train'] # Note there is no test in this dataset only train so we treated as if it is the complete dataset

Dataset({
    features: ['text', 'label', 'label_text', 'idx', 'query_idx', 'positive_idx', 'negative_idx'],
    num_rows: 16000
})

### Split The Data

In [107]:
X_train, X_test, y_train, y_test = train_test_split(ds['train']['text'], ds['train']['label'], test_size=0.2, random_state=42)

In [108]:
len(X_train), len(X_test), len(y_train), len(y_test)

(12800, 3200, 12800, 3200)

### Data Preprocessing

In [109]:
def process_and_tokenize(x):
    arr_tokens = [word_tokenize(sentence.lower()) for sentence in x]
    for i, tokens in enumerate(arr_tokens):
       arr_tokens[i] = [token for token in tokens if token.isalpha()]

    return arr_tokens

### Lemmatization

In [110]:
def smart_lemmatize(word):
    verb_synsets = wordnet.synsets(word, pos='v')
    if verb_synsets:
        return lemmatizer.lemmatize(word, pos='v')
    return lemmatizer.lemmatize(word, pos='n')

def lemmatization(x):
    return [[smart_lemmatize(token) for token in tokens] for tokens in x]

In [111]:
X_train_tokenized = lemmatization(process_and_tokenize(X_train))

In [151]:
X_train_tokenized[1]

['i',
 'do',
 'not',
 'know',
 'be',
 'that',
 'she',
 'be',
 'of',
 'the',
 'damn',
 'and',
 'that',
 'she',
 'have',
 'have',
 'century',
 'to',
 'hone',
 'the',
 'very',
 'word',
 'she',
 'wield',
 'against',
 'me',
 'with',
 'their',
 'razor',
 'edge',
 'in',
 'hindsight',
 'i',
 'can',
 'not',
 'help',
 'but',
 'feel',
 'resign',
 'to',
 'the',
 'fate',
 'that',
 'inevitably',
 'follow',
 'for',
 'i',
 'be',
 'helpless',
 'to',
 'withstand',
 'her']

## 2.2: Logistic Regression

#### Feature Representation

In [140]:
def to_bi_grams(x):
    vocab = {}
    for tokens in x:
        for bg in list(bigrams(tokens)):
            if bg not in vocab:
                vocab[bg] = len(vocab)

    return vocab

In [141]:
def construct_sparse_binary_vector(vocab, x):
    rows = []
    cols = []

    for row_idx, tokens in enumerate(x):
        for bg in set(bigrams(tokens)):
            col_idx = vocab.get(bg)
            if col_idx is not None:
                rows.append(row_idx)
                cols.append(col_idx)

    data = np.ones(len(rows), dtype=np.int8)

    return csr_matrix((data, (rows, cols)), shape=(len(x), len(vocab)), dtype=np.int8)

In [142]:
vocab = to_bi_grams(X_train_tokenized)
X_train_sparse = construct_sparse_binary_vector(vocab, X_train_tokenized)

In [147]:
X_train_sparse.shape

(12800, 80933)

In [153]:
X_train_csr = csr_matrix(X_train_sparse)
X_train_np = np.array(X_train_sparse)

np.int8(0)

#### Algorithm Implementation